# (Quick) Python introduction 2 and cell temperature / DC power

<table>
  <tr>
    <td><img src="https://raw.githubusercontent.com/Mohamed-ENNHIRI/ENER958/main/notebooks/logos/solaracademy.png" width="200" />
    <td><img src="https://raw.githubusercontent.com/Mohamed-ENNHIRI/ENER958/main/notebooks/logos/usmb.png" width="200" />
    <td><img src="https://raw.githubusercontent.com/Mohamed-ENNHIRI/ENER958/main/notebooks/logos/cnrs.png" width="80" />
  </tr>
</table>

<sub>Author: Mohamed Ennhiri, PhD student, LOCIE UMR CNRS 5271, Université Savoie Mont Blanc</sub>

<sub>Structure and format inspired by the course given by Alexandre Mathieu at the Solar Academy in 2024, "PV failure detection based on operational time series". Physics and data: pvlib, PVGIS, ODRE, RTE.</sub>

This notebook continues the python introduction and models the module temperature and the DC power of a PV installation.

It is broken down into two parts:

1. **Python introduction**: filtering, "if" and loops, plots, functions. The student is invited to execute the cells and do the exercices.
2. **Cell temperature and DC power exercices**: The student models the module temperature and the DC power for the worked hour of the course, then for a whole year.

The students are expected to do the **exercices**.

## I. Python introduction

In this introduction, 4 topics are tackled:

1. Filtering over a pandas Series or DataFrame
2. "If" and loops commands
3. Plots
4. Functions

In [ ]:
# Please execute this cell to import packages
import numpy as np # For math operations
import pandas as pd # To create indexed time series
import matplotlib.pyplot as plt # To plot figures

# In Google Colab, pvlib is not installed by default: the line below installs it (takes 30 s)
!pip install -q pvlib
import pvlib
print("pvlib", pvlib.__version__)

In [ ]:
# Load the satellite weather data of the course site from an online file: GHI, DNI, DHI, air temperature, wind speed
# Site: Savoie, near Le Bourget-du-Lac (Lat 45.64 N, Long 5.92 E). Source: PVGIS SARAH3, year 2023, hourly.
urlw = "https://raw.githubusercontent.com/Mohamed-ENNHIRI/ENER958/main/data/weather_45.64N_5.92E_2023.csv"
weather_data = pd.read_csv(urlw, index_col=0) # Import the csv file and use the first column as index
weather_data.index = pd.to_datetime(weather_data.index) # Convert the index into datetimes (UTC)
print(weather_data.head(5)) # Show the first five lines

In [ ]:
# Load the solar position (zenith, azimuth, elevation) calculated with the NREL algorithm:
# I. Reda and A. Andreas, Solar position algorithm for solar radiation applications. Solar Energy, 76(5), 2004.
urls = "https://raw.githubusercontent.com/Mohamed-ENNHIRI/ENER958/main/data/solarpos_45.64N_5.92E_2023.csv"
solar_position = pd.read_csv(urls, index_col=0)
solar_position.index = pd.to_datetime(solar_position.index)
print(solar_position.head(5))

### I.1 Pandas filtering

When it comes to making calculations faster or selecting a specific chunk of the DataFrame, filtering is useful.

In [ ]:
# Filter the DataFrame when ghi is over a threshold
filter = weather_data["ghi"] > 800 # A Series of True / False
print(weather_data.loc[filter].head(5)) # Keep the rows where the filter is True

In [ ]:
# Sometimes we do not see much about the data and it is interesting to filter on some dates
filter_date = weather_data.index > pd.to_datetime("2023-12-01").tz_localize("UTC") # After the 1st of December
weather_data.loc[filter_date, "ghi"].plot() # Plot only one column

In [ ]:
# The filters can add up
filter_all = filter | filter_date # "or"
filter_all2 = filter & filter_date # "and"
print(filter_all2.sum()) # Number of hours with ghi > 800 W/m2 in December: the sum of a True / False Series counts the True

#### Exercice

Apply the following filters on `weather_data` with the condition "and":

- ghi > 800 W/m²
- between the 1st of June and the 1st of August

and plot `dhi` only, with markers.

In [ ]:
filter_ghi = weather_data["ghi"] > 800
filter_period = (weather_data.index >= pd.to_datetime("2023-06-01").tz_localize("UTC")) & (weather_data.index < pd.to_datetime("2023-08-01").tz_localize("UTC"))
weather_data.loc[filter_ghi & filter_period, "dhi"].plot(marker="o", linewidth=0, ylabel="DHI [W/m²]")

### I.2 "If" and loops

"If" allows to verify an assertion and execute actions. Loops enable to repeat actions over all elements of a list or a DataFrame.

In [ ]:
a = 1
if a < 0: # Assertion: is "a" under 0 ? Do not forget the ":" at the end of the line
    print("a is lower than 0") # Line not executed since the assertion is wrong. Do not forget the "tab" indentation after an "if"
else:
    print("a is greater than or equal to 0")

In [ ]:
# Loop over all elements of a list
for element in ["a", "b"]:
    print(element)

# Loop over the first rows of the DataFrame one by one (slow on 8760 rows, prefer column operations when possible)
for index, row in weather_data.head(3).iterrows():
    print(index, row["ghi"] + row["dhi"])

#### Exercice

Create a new pd.Series with the same index as `weather_data`, equal to 1 when ghi > 100 W/m² (daytime) and 0 otherwise. Print the number of daytime hours of the year.

In [ ]:
# The fast way: a True / False Series converted into integers
daytime = (weather_data["ghi"] > 100).astype(int)
print(f"Daytime hours: {daytime.sum()}")

### I.3 Plots

In [ ]:
# Make a figure with two axes (two figures top / bottom) which share the same x axis
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6), sharex=True)

# Show the air temperature on the first axis "ax1" with a title and a label
weather_data["temp_air"].plot(ax=ax1, title="Air temperature [°C] as function of time", ylabel="Temperature [°C]")

# Show the three irradiance components on the second axis
weather_data[["ghi", "dni", "dhi"]].plot(ax=ax2, title="Irradiance [W/m²] as function of time", ylabel="Irradiance [W/m²]")
plt.tight_layout()

In [ ]:
# A scatter plot with matplotlib, with the colour given by a third variable
fig, ax = plt.subplots(figsize=(9, 4))
sc = ax.scatter(weather_data["ghi"], weather_data["dhi"], c=weather_data.index.month, s=3) # s is the marker size
plt.colorbar(sc, label="month")
ax.set_xlabel("GHI [W/m²]")
ax.set_ylabel("DHI [W/m²]")

### I.4 Functions

Functions enable to package small algorithms which are easy to apply and to re-use.

In [ ]:
def cos_aoi_function(zenith, s_azimuth, beta, azimuth):
    """Cosine of the angle of incidence. All angles in degrees."""
    cos_aoi = np.cos(np.radians(zenith)) * np.cos(np.radians(beta)) + np.sin(np.radians(zenith)) * np.sin(np.radians(beta)) * np.cos(np.radians(s_azimuth - azimuth))
    return cos_aoi

# Apply the function with the values of the worked hour, returns 0.9867
print(cos_aoi_function(22.64, 193.3, 30, 180))

#### Exercice

Apply the function `cos_aoi_function()` on the pd.DataFrame `solar_position` for a tilt of 30° and a south azimuth, clip the result at 0, and count the hours where cos(AOI) was negative before clipping.

In [ ]:
cos_aoi = cos_aoi_function(solar_position["zenith"], solar_position["azimuth"], 30, 180)
print(f"Hours with the sun behind the plane: {(cos_aoi < 0).sum()}")
cos_aoi = cos_aoi.clip(lower=0)

## II. Cell temperature and DC power

The hotter a module is, the less efficient it is ! Three models of the module temperature are seen in the course:

- NOCT (Ross): $T_c = T_a + \frac{NOCT - 20}{800} \cdot G_{POA}$, with NOCT = 45 °C
- Faiman: $T_c = T_a + \frac{G_{POA}}{U_0 + U_1 \cdot WS}$, with $U_0$ = 25.0 W/(m²K) and $U_1$ = 6.84 W/(m³Ks), open rack
- SAPM: $T_m = G_{POA} \cdot e^{a + b \cdot WS} + T_a$ and $T_c = T_m + \frac{G_{POA}}{1000} \Delta T$, with a = −3.56, b = −0.075, ΔT = 3 K, open rack glass-polymer

And the DC power with the PVWatts linear model: $P_{DC} = P_0 \cdot \frac{G_{POA}}{1000} \cdot [1 + \gamma_P (T_c - 25)]$

### II.1 The worked hour of the course

Same inputs as the course slide: POA = 962.7 W/m², Ta = 26 °C, wind 2 m/s, P0 = 3 kWp, γP = −0.35 %/K.

#### Exercice

Write three functions `noct_model(ta, poa)`, `faiman_model(ta, poa, ws)` and `sapm_model(ta, poa, ws)`, apply them to the worked hour and print the three cell temperatures. Expected with Faiman: 50.9 °C.

In [ ]:
def noct_model(ta, poa, noct=45):
    return ta + (noct - 20) / 800 * poa

def faiman_model(ta, poa, ws, u0=25.0, u1=6.84):
    return ta + poa / (u0 + u1 * ws)

def sapm_model(ta, poa, ws, a=-3.56, b=-0.075, delta_t=3):
    t_mod = poa * np.exp(a + b * ws) + ta # Module back-surface temperature
    return t_mod + poa / 1000 * delta_t # Cell temperature

poa_td, ta_td, ws_td = 962.7, 26, 2
print(f"NOCT: {round(noct_model(ta_td, poa_td), 1)} °C")
print(f"Faiman: {round(faiman_model(ta_td, poa_td, ws_td), 1)} °C")
print(f"SAPM: {round(sapm_model(ta_td, poa_td, ws_td), 1)} °C")

#### Exercice

Write the function `pvwatts_dc(poa, t_cell, p0, gamma_pdc)` and apply it to the worked hour with the Faiman temperature. Expected: 2626 W.

In [ ]:
def pvwatts_dc(poa, t_cell, p0, gamma_pdc):
    """PVWatts linear model. p0 in W, gamma_pdc per °C (-0.0035 for -0.35 %/K). Returns W."""
    return p0 * poa / 1000 * (1 + gamma_pdc * (t_cell - 25))

p_dc_td = pvwatts_dc(poa_td, faiman_model(ta_td, poa_td, ws_td), 3000, -0.0035)
print(f"DC power: {round(p_dc_td)} W")

### II.2 One year

The POA irradiance over the year is calculated the lazy way, then the three temperature models and the DC power are compared.

In [ ]:
from pvlib.irradiance import get_total_irradiance
beta, azimuth, rho = 30, 180, 0.2
poa_data = get_total_irradiance(beta, azimuth, solar_position["zenith"], solar_position["azimuth"], weather_data["dni"], weather_data["ghi"], weather_data["dhi"], albedo=rho)
poa_global = poa_data["poa_global"]

#### Exercice

Apply the three temperature models over the year, plot the first four days of July on one figure, then calculate the DC power with each of them and print the three annual DC energies in kWh. How much does the model choice change the annual energy, in percent?

In [ ]:
t_noct = noct_model(weather_data["temp_air"], poa_global)
t_faiman = faiman_model(weather_data["temp_air"], poa_global, weather_data["wind_speed"])
t_sapm = sapm_model(weather_data["temp_air"], poa_global, weather_data["wind_speed"])

fig, ax = plt.subplots(figsize=(9, 4))
t_noct.loc["2023-07-01":"2023-07-04"].plot(ax=ax, label="NOCT")
t_faiman.loc["2023-07-01":"2023-07-04"].plot(ax=ax, label="Faiman")
t_sapm.loc["2023-07-01":"2023-07-04"].plot(ax=ax, label="SAPM")
plt.ylabel("Cell temperature [°C]")
plt.legend()

energies = {}
for name, t_cell in [("NOCT", t_noct), ("Faiman", t_faiman), ("SAPM", t_sapm)]:
    energies[name] = pvwatts_dc(poa_global, t_cell, 3000, -0.0035).sum() / 1000 # kWh
    print(f"{name}: {round(energies[name], 1)} kWh")
print(f"Spread between models: {round(100 * (max(energies.values()) / min(energies.values()) - 1), 2)} %")

#### Exercice, to go further

Calculate the annual DC energy with γP = 0 (no temperature effect). The difference with the Faiman case is the thermal loss of the year, in percent.

In [ ]:
e_gamma0 = pvwatts_dc(poa_global, t_faiman, 3000, 0.0).sum() / 1000
print(f"Thermal loss: {round(100 * (1 - energies['Faiman'] / e_gamma0), 2)} %")